In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.home_credit_bronze.application_train")

# 1. literal 'null' strings
df.filter(F.col("OWN_CAR_AGE") == "null").count()

# 2. DAYS_EMPLOYED sentinel
df.groupBy("DAYS_EMPLOYED").count().orderBy(F.desc("count")).show(5)

# 3. unknown-value tokens
df.groupBy("CODE_GENDER").count().show()
df.groupBy("ORGANIZATION_TYPE").count().orderBy(F.desc("count")).show(5, truncate=False)

# 4. flag values
df.groupBy("FLAG_OWN_CAR", "FLAG_MOBIL").count().show()

# 5. EXT_SOURCE_2 range
df.select(
    F.min(F.expr("try_cast(EXT_SOURCE_2 AS DOUBLE)")),
    F.max(F.expr("try_cast(EXT_SOURCE_2 AS DOUBLE)")),
).show()

In [0]:
df.filter((F.col("DAYS_EMPLOYED") == "365243") & (F.col("ORGANIZATION_TYPE") == "XNA")).count()

In [0]:
df.filter(F.col("OWN_CAR_AGE") == "null").count()
df.filter(F.col("DAYS_EMPLOYED") == "365243") \
  .groupBy("NAME_INCOME_TYPE").count().orderBy(F.desc("count")).show()

In [0]:
df = spark.table("workspace.home_credit_bronze.bureau")
df.limit(5).display()

In [0]:
b = spark.table("workspace.home_credit_bronze.bureau")
a = spark.table("workspace.home_credit_bronze.application_train")

b.groupBy("CREDIT_ACTIVE").count().show()
b.groupBy("CREDIT_CURRENCY").count().show()
b.groupBy("CREDIT_TYPE").count().orderBy(F.desc("count")).show(truncate=False)

# key uniqueness
print(b.count(), b.select("SK_ID_BUREAU").distinct().count())

# orphans: bureau customers missing from application_train
b.select("SK_ID_CURR").distinct().join(a.select("SK_ID_CURR"), "SK_ID_CURR", "left_anti").count()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.home_credit_silver.customer UNION ALL            -- expect 307,511
SELECT COUNT(*) FROM workspace.home_credit_silver.customer_quarantine UNION ALL  -- expect 0
SELECT COUNT(*) - COUNT(DISTINCT sk_id_curr) AS duplicate_keys
FROM workspace.home_credit_silver.customer;    

In [0]:
s = spark.table("workspace.home_credit_silver.customer")

# sentinel gone, is_employed correct (expect 55,374 false)
s.filter("is_employed = false").count()
s.filter("days_employed = 365243").count()          # expect 0

# XNA gone (expect 0 and 0)
s.filter("code_gender = 'XNA'").count()
s.filter("organization_type = 'XNA'").count()

# types and lowercase names look right
s.printSchema()
s.select("sk_id_curr", "age_years", "years_employed", "is_employed", "flag_own_car").show(5)

In [0]:
df=spark.table("workspace.home_credit_bronze.bureau")

df.limit(5).display()